In [41]:
import pandas as pd
import sys
import os
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

## Make exported plots' text seen as text in Illustrator
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

sns.set_theme(style="white")

palette = {"EPcadKO"                    : "#d55e00",
           "EPcadKO+ER-StayGold+Ecad-deltaBCBS-RFP" : "#cc78bc",
           "EPcadKO+ER-StayGold+Ecad-GGG-RFP"       : "#56b4e9",
           "EPcadKO+ER-StayGold+Ecad-WT-RFP"        : "#029e73",
           "A431"                       : "#0173b2",
           "puncta" : "#56b4e9",
           "linear" : "#cc78bc",
           
          }

In [42]:
sns.color_palette("colorblind")

[(0.00392156862745098, 0.45098039215686275, 0.6980392156862745),
 (0.8705882352941177, 0.5607843137254902, 0.0196078431372549),
 (0.00784313725490196, 0.6196078431372549, 0.45098039215686275),
 (0.8352941176470589, 0.3686274509803922, 0.0),
 (0.8, 0.47058823529411764, 0.7372549019607844),
 (0.792156862745098, 0.5686274509803921, 0.3803921568627451),
 (0.984313725490196, 0.6862745098039216, 0.8941176470588236),
 (0.5803921568627451, 0.5803921568627451, 0.5803921568627451),
 (0.9254901960784314, 0.8823529411764706, 0.2),
 (0.33725490196078434, 0.7058823529411765, 0.9137254901960784)]

In [43]:
print(sns.color_palette("colorblind").as_hex())

['#0173b2', '#de8f05', '#029e73', '#d55e00', '#cc78bc', '#ca9161', '#fbafe4', '#949494', '#ece133', '#56b4e9']


In [44]:
input_csv_dir = r"C:\data\Sonam\2024-05-30_MAPPER_Quantification\05_csvs"
output_csv_dir = r"C:\data\Sonam\2024-05-30_MAPPER_Quantification\06_merged-csv"

output_csv_dir_exists = os.path.exists(output_csv_dir)

if not output_csv_dir_exists:
    os.makedirs(output_csv_dir)

# example filename A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_001-Denoised_R1_Object_Predictions.tif_ROI-0_Class-1_Results.csv

foo = "A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_001-Denoised_R1_Object_Predictions.tif_ROI-0_Class-1_Results.csv"

split1, split2 = foo.split(".tif_")
print(split1, split2)
rep = split1.split("_")[-3][-1]
obj_class = split2.split("_")[1][-1]
roi_ID = split2.split("_")[0].split("-")[-1]
print(roi_ID)
print(obj_class)
print(rep)
cell = split1.split("_")[-4].split("-")[0]
print(cell)
print(split1.split("_"))
print(split2.split("_"))
split1.rsplit("_", 4)[0]
#split3, split4 = split1.split("_")
#print(split3, split4)

In [45]:
def ProcessDF(df):
    
    # Example filename A431+ER-StayGold_WGA+ER-000_cell-001.tif_C1_Intensity.csv
    
    df[["split1", "split2"]] = df["Label"].str.split(".tif_", expand = True)

    # deal with the future warning by initializing first
    # https://stackoverflow.com/questions/77098113/solving-incompatible-dtype-warning-for-pandas-dataframe-when-setting-new-column
    df["Type"] = None

    df.loc[df["ObjectClass"] == 1, "Type"] = "puncta"
    df.loc[df["ObjectClass"] == 2, "Type"] = "linear"
    
    df["cell_line"] = df["split1"].str.rsplit(pat = "_", n = 4, expand=True)[0]
    df["cell_line"] = df["cell_line"].replace(
        {"A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA"    : "A431",
         "EPcadKO+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA" : "EPcadKO"
        })
    df["Rep"] = df["split1"].str.split("_", expand=True).iloc[:,-3].str.slice(start=-1)

    df.drop(columns = ["split1", "split2"], inplace = True)
    
    return df

input_CSVs = os.listdir(input_csv_dir)

df_concat = pd.DataFrame()

for f in input_CSVs:
    f_path = os.path.join(input_csv_dir, f)
    #print(f_path)
    # Not all ROIs have both object classes, but empty csvs (except for some headers) were saved regardless.
    # Without try/except, you'd get a valueerror due to missing columns
    try:
        df_temp = pd.read_csv(f_path, 
                              usecols = 
                              ["Label", "Area", "Mean", "Feret", "IntDen", "RawIntDen",
                               "MinFeret", "ObjectClass", "ParticleID", "Image", "ROI", 
                               "ROI_Area", "ROI_Feret"]
                             )
        df_temp["Label"] = f
        df_processed = ProcessDF(df_temp)
        df_concat = pd.concat([df_concat, df_processed])
    except:
        continue

df = df_concat
df

,Label,Area,Mean,Feret,IntDen,RawIntDen,MinFeret,ObjectClass,ParticleID,Image,ROI,ROI_Area,ROI_Feret,Type,cell_line,Rep
0,A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_0...,0.013,132.333,4.243,1.677,397.0,1.414,1,0,A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_0...,0,116.842,25.684,puncta,A431,1
1,A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_0...,0.017,116.250,2.828,1.965,465.0,2.000,1,1,A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_0...,0,116.842,25.684,puncta,A431,1
2,A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_0...,0.085,123.000,6.708,10.394,2460.0,4.919,1,2,A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_0...,0,116.842,25.684,puncta,A431,1
3,A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_0...,0.131,170.935,8.062,22.388,5299.0,5.657,1,3,A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_0...,0,116.842,25.684,puncta,A431,1
4,A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_0...,0.186,132.909,10.817,24.708,5848.0,6.261,1,4,A431+ER-StayGold+MAPPER-mApple_MAPPER+ER+WGA_0...,0,116.842,25.684,puncta,A431,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11,EPcadKO+ER-StayGold+MAPPER-mApple_MAPPER+ER+WG...,0.097,119.609,8.944,11.623,2751.0,4.000,1,11,EPcadKO+ER-StayGold+MAPPER-mApple_MAPPER+ER+WG...,2,23.592,14.450,puncta,EPcadKO,1
12,EPcadKO+ER-StayGold+MAPPER-mApple_MAPPER+ER+WG...,0.106,151.800,7.616,16.034,3795.0,4.000,1,12,EPcadKO+ER-StayGold+MAPPER-mApple_MAPPER+ER+WG...,2,23.592,14.450,puncta,EPcadKO,1
13,EPcadKO+ER-StayGold+MAPPER-mApple_MAPPER+ER+WG...,0.292,164.319,12.369,47.903,11338.0,7.000,1,13,EPcadKO+ER-StayGold+MAPPER-mApple_MAPPER+ER+WG...,2,23.592,14.450,puncta,EPcadKO,1
14,EPcadKO+ER-StayGold+MAPPER-mApple_MAPPER+ER+WG...,0.059,115.500,5.000,6.832,1617.0,4.000,1,14,EPcadKO+ER-StayGold+MAPPER-mApple_MAPPER+ER+WG...,2,23.592,14.450,puncta,EPcadKO,1


## Split by object classification type

In [46]:
def groupby_for_objects(df, split_by_type=True):
    if split_by_type:
        cols = ["cell_line","Type", "Rep", "Image", "ROI"]
    else:
        cols = ["cell_line","Rep", "Image", "ROI"]
    
    count_name = "Count_separated_by_type-{}".format(split_by_type)
    total_obj_area_name = "Total_Object_Area_separated_by_type-{}".format(split_by_type)
    
    df_count = df.groupby(cols).size().to_frame(name=count_name).reset_index()
    df_obj_sum = df.groupby(cols)["Area"].sum().to_frame(name=total_obj_area_name).reset_index()
    df_ROI_Area = df.groupby(cols)["ROI_Area"].unique().to_frame().reset_index()
    df_ROI_Area["ROI_Area"] = df_ROI_Area["ROI_Area"].str[0]
    df_ROI_Feret = df.groupby(cols)["ROI_Feret"].unique().to_frame().reset_index()
    df_ROI_Feret["ROI_Feret"] = df_ROI_Feret["ROI_Feret"].str[0]
    df_main = df_count.merge(df_ROI_Feret).merge(df_ROI_Area).merge(df_obj_sum)
    
    df_main["Count_per_ROI_Area"] = df_main[count_name]/df_main["ROI_Area"]
    df_main["Obj_Area_per_ROI_Area"] = df_main[total_obj_area_name]/df_main["ROI_Area"]
    df_main["Count_per_ROI_Feret"] = df_main[count_name]/df_main["ROI_Feret"]
    df_main["Obj_Area_per_ROI_Feret"] = df_main[total_obj_area_name]/df_main["ROI_Feret"]
    df_main["Count_per_scaled_distance"] = (df_main[count_name] * df_main["ROI_Feret"])/(df_main["ROI_Area"]**2)
    df_main["Obj_Area_per_scaled_distance"] = (df_main[total_obj_area_name] * df_main["ROI_Feret"])/(df_main["ROI_Area"]**2)
    
    return df_main

In [47]:
df_by_type = groupby_for_objects(df, True)
df_by_type.to_csv(output_csv_dir + "/" + "DataFrame_with_type.csv")

PermissionError: [Errno 13] Permission denied: 'C:\\data\\Sonam\\2024-05-30_MAPPER_Quantification\\06_merged-csv/DataFrame_with_type.csv'

In [ ]:
df_without_type = groupby_for_objects(df, False)
df_without_type.to_csv(output_csv_dir + "/" + "DataFrame_without_type.csv")


In [ ]:
sns.displot(data=df_by_type,  col="cell_line",  x = "Obj_Area_per_ROI_Area",
            hue="Type", element="step", palette=palette#multiple="stack"
           )

#plt.savefig("Total-Obj-Area_per_ROI-Area_split-by-type-True.png")

In [ ]:
sns.displot(data=df_by_type,  col="cell_line",  x = "Count_per_ROI_Area",
            hue="Type", element="step",palette=palette,
            #multiple="stack"
           )


In [ ]:
sns.displot(data=df_by_type,  col="Type",  x = "Count_per_ROI_Feret",
            hue="cell_line", element="step",palette=palette,
           )
plt.savefig("Count_per_ROI-Feret_split-by-type-True_stacked-histograms.png")
#multiple="stack"

In [ ]:
sns.histplot(data=df_by_type, x = "Count_per_ROI_Area", hue="cell_line", common_norm=True,element="step", fill=True,
             palette=palette
            )
#sns.displot(data=df_by_type, x = "Count_per_ROI_Area", hue="cell_line", kde=False)
#plt.savefig("Count_per_ROI-Area_split-by-type-False_not-stacked.png")

In [ ]:
sns.displot(data=df_without_type,  hue="cell_line",  x = "Obj_Area_per_ROI_Area",element="step",
           palette=palette)
plt.savefig("Total-Obj-Area_per_ROI-Area_split-by-type-False.png")

In [ ]:
sns.histplot(data=df_by_type, x = "Count_per_ROI_Area", hue="cell_line", common_norm=True,element="step", fill=True)

In [ ]:
sns.histplot(df_without_type, x="Count_per_ROI_Area",element="step", hue="cell_line")

In [ ]:
sns.histplot(df_without_type, x="Obj_Area_per_ROI_Area",element="step", hue="cell_line",)
plt.savefig("Total-Obj-Area_per_ROI-Area_split-by-type-False.png")

In [ ]:
sns.histplot(df_without_type, x="Obj_Area_per_ROI_Feret",element="step", hue="cell_line")
plt.savefig("Total-Obj-Area_per_ROI-Feret_split-by-type-False.png")

In [ ]:
sns.histplot(df_without_type, x="Count_per_ROI_Feret", element="step",hue="cell_line")
plt.savefig("Count_per_ROI-Area_split-by-type-False.png")

In [ ]:
sns.histplot(df_without_type, x="Count_per_scaled_distance",element="step", hue="cell_line")


In [ ]:
sns.histplot(df_without_type, x="Obj_Area_per_scaled_distance",element="step", hue="cell_line")

In [ ]:
df_count = df.groupby(["cell_line","Type", "Rep", "Image", "ROI"]).size().to_frame(name="Count_separated_by_type").reset_index()

In [ ]:
df_obj_sum = df.groupby(["cell_line","Type", "Rep", "Image", "ROI"])["Area"].sum().to_frame().reset_index()
df_obj_sum

In [ ]:
df_ROI_Area = df.groupby(["cell_line","Type", "Rep", "Image", "ROI"])["ROI_Area"].unique().to_frame().reset_index()
df_ROI_Area["ROI_Area"] = df_ROI_Area["ROI_Area"].str[0]
df_ROI_Area

In [ ]:
df_ROI_Feret = df.groupby(["cell_line","Type", "Rep", "Image", "ROI"])["ROI_Feret"].unique().to_frame().reset_index()
df_ROI_Feret["ROI_Feret"] = df_ROI_Feret["ROI_Feret"].str[0]
df_ROI_Feret

In [ ]:
df_main = df_count.merge(df_ROI_Feret).merge(df_ROI_Area).merge(df_obj_sum)
df_main

In [ ]:
df_main.groupby

In [ ]:
df.groupby(["cell_line"]).size()

In [ ]:
df["cell_line"].unique()

In [ ]:
df["Rep"].unique()

In [ ]:
#df["cell_line"] = df["cell_line"].replace({"A431+ER-StayGold_WGA+ER" : "A431+ER-StayGold",
#                                           "EPcadKO+ER-StayGold_WGA+ER" : "EPcadKO+ER-StayGold",
#                                           "EPcadKO+ER-StayGold+Ecad-WT-RFP_WGA+ER+Ecad" : "EPcadKO+ER-StayGold+Ecad-WT-RFP"
#                                          }
#                                         )

In [ ]:
mean_list = list()
cell_line_list = list()
cell_list = list()
image_list = list()
rep_list = list()

for key, grp in df_ER.groupby(["cell_line", "cell", "image","Rep"])["Mean"]:
    (cell_line_temp, cell_temp, image_temp, rep_temp) = key
    cell_line_list.append(cell_line_temp)
    cell_list.append(cell_temp)
    image_list.append(image_temp)
    rep_list.append(rep_temp)
    
    
    # The first (index 0) measurement is the nucleus,
    # so we want the second measurement (index 1) 
    mean_list.append(grp / grp.iloc[1])
foo = pd.DataFrame(list(zip(cell_line_list, cell_list, image_list, rep_list, mean_list)),
                  columns = ["cell_line", "cell", "image", "Rep", "Normalized Mean"]
                  )



In [ ]:
foo = foo.explode("Normalized Mean", ignore_index=True)
foo

In [ ]:
foo["Slice"] = foo.groupby(["cell_line", "cell", "image", "Rep"]).cumcount()+1
foo

In [ ]:
foo.to_csv(output_csv_dir + "2024-03-26_normalized_ER_rings-to-second-ring.csv", index=False)

In [ ]:
sns.set_theme(style="white", palette=None)
sns.set(rc = {'figure.figsize':(20,10)})
sns.set(font_scale=2.25)
b = sns.lineplot(data = foo,
                 x = "Slice",
                 y = "Normalized Mean",
                 hue = "cell_line",
                 markers = True,
                 dashes = False,
                 palette=palette,
                )
(b.set(xlim=(-.5, 24),
       xticks=[4, 20],
      )
)
b.set_xticklabels(["Perinuclear", "Peripheral"])
plt.ylabel("Normalized ER Mean Intensity")
plt.xlim(1,24)
plt.ylim(0, 1.1)
plt.xlabel("")
 #.set_ylabels("Normalized Mean Intensity")
 #.set_xlabels("Ring #")

plt.legend(title = "Cell line")
plt.savefig(output_csv_dir + "plot_lines.png", bbox_inches = "tight")
plt.savefig(output_csv_dir + "plot_lines.pdf", bbox_inches = "tight")

In [ ]:
sns.set_theme(style="white")
sns.set(rc = {'figure.figsize':(20,10)})
sns.set(font_scale=1.25)

c = sns.catplot(
    data = foo,
    x = "Slice",
    y = "Normalized Mean",
    hue = "cell_line",
    errorbar = "se", 
    kind="point",
    fillstyle="none",
    marker=None,
    palette=palette,
    #legend=False,
)
#sns.move_legend(c, "lower left", 
                #bbox_to_anchor=(.5, 1),
#                title=None)
c.set(xlim=(1, 24), 
      #xticks=[0, 4, 8, 12, 16, 20, 24])
       xticks=[5, 20],
     )
#c.set_xticklabels([0, 4, 8, 12, 16, 20, 24])
c.set_xticklabels(["Perinuclear", "Peripheral"])

#c.ax.legend(loc=1)
#plt.legend(title = "Cell line", 
#           loc = "lower left",
#          frameon=False)
plt.ylabel("Normalized ER Mean Intensity")
#plt.xlabel("Relative radial position")
plt.xlabel("")
plt.xlim(1,24)
plt.ylim(0, 1.1)

plt.savefig(output_csv_dir + "plot_lineplot-with-markers.png", bbox_inches = "tight", dpi= 300)
plt.savefig(output_csv_dir + "plot_lineplot-with-markers.pdf", bbox_inches = "tight", dpi= 300)

In [ ]:
sharp = sns.catplot(foo,
                    x = "Slice",
                    y = "Normalized Mean",
                    hue = "Rep",
                    col = "cell_line",
                    kind = "point",
                    #row = "Rep"
                   )

In [ ]:
sharp = sns.catplot(df_ER,
                    x = "Slice",
                    y = "Mean",
                    hue = "Rep",
                    col = "cell_line",
                    kind = "point",
                    #row = "Rep"
                   )

In [ ]:
df_WGA = df.loc[df["channel_name"] == "WGA"]

sharp = sns.catplot(df_WGA,
                    x = "Slice",
                    y = "Mean",
                    hue = "Rep",
                    col = "cell_line",
                    kind = "point",
                    #row = "Rep"
                   )

In [ ]:
#df_ER.groupby(["cell_line", "cell", "image"])[["Mean"]].nth(1)

In [ ]:
non_nuclear_ER_rings = df_ER.groupby(["cell_line", "cell", "image", "Rep"])[["Mean"]].tail(23)
non_nuclear_ER_rings

In [ ]:
nuclear_ER_rings = df_ER.groupby(["cell_line", "cell", "image", "Rep"])[["Mean"]].nth(1)
nuclear_ER_rings


In [ ]:
non_nuclear_ER_rings = df_ER.loc[(df_ER["Slice"] > 1)]
non_nuclear_ER_rings

In [ ]:
df_ER.loc[(df_ER["Slice"] == 1)]

In [ ]:
scaled_non_nuclear_ER_rings = non_nuclear_ER_rings / df_ER.loc[(df_ER["Slice"] == 1)]
scaled_non_nuclear_ER_rings

In [ ]:
foo = df_ER.loc[(df_ER["Slice"] > 1) & (df_ER["Slice"] < 24)]
foo

In [ ]:
inner = foo[["cell_line", "Mean"]].groupby("cell_line").mean()
inner

In [ ]:
inner = foo[["cell_line", "Mean", "Rep"]].groupby(["cell_line", "Rep"]).mean()
inner

In [ ]:
outer = df_ER.loc[(df_ER["Slice"] == 24)][["cell_line", "Mean", "Rep"]].groupby(["cell_line", "Rep"]).mean()
outer

In [ ]:
outer = df_ER.loc[(df_ER["Slice"] == 24)][["cell_line", "Mean"]].groupby("cell_line").mean()
outer

In [ ]:
relative_change = outer / inner
relative_change

In [ ]:
relative_change.to_csv(output_csv_dir + "outer-to-inner-ratio.csv", index=False)

In [ ]:
df.to_csv(output_csv_dir + "2024-03-26_merged-results.csv", index=False)